In [ ]:
#uv add -r rag-requirements.txt

Resolved 261 packages in 2ms
Checked 232 packages in 552ms


In [36]:
# PDF 읽고 페이지 구분 및 Part 구분
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "data/practice.pdf"

loader = PyMuPDFLoader(file_path)

docs = loader.load()

# 맨 끝의 설문조사 제외하기 위해 범위 설정
docs = docs[1:28]

for doc in docs:

    page = doc.metadata["page"]+1

    if page >= 2 and page <= 4:
        doc.metadata["part"] = "01"
        doc.metadata["chapter"] = "생성형 AI란?"
        doc.metadata["keyword"] = "생성형 AI"

    elif page >= 5 and page <= 9:
        doc.metadata["part"] = "02"
        doc.metadata["chapter"] = "저작권"
        doc.metadata["keyword"] = "저작권"       

    elif page >= 10 and page <= 14:
        doc.metadata["part"] = "03"
        doc.metadata["chapter"] = "책임성"
        doc.metadata["keyword"] = "책임"       
    elif page >= 15 and page <= 19:
        doc.metadata["part"] = "04"
        doc.metadata["chapter"] = "허위조작정보"
        doc.metadata["keyword"] = "허위조작"       
    elif page >= 20 and page <= 23:
        doc.metadata["part"] = "05"
        doc.metadata["chapter"] = "개인정보 인격권"
        doc.metadata["keyword"] = "개인정보"       
    elif page >= 24 and page <= 28:
        doc.metadata["part"] = "06"
        doc.metadata["chapter"] = "오남용"
        doc.metadata["keyword"] = "오남용"

In [37]:
# 소제목 구분

import re

def is_heading(line, chapter):
    line = line.strip()

    if not line:
        return False
    if line == chapter :
        return False
    if line == "관련사례":
        return True
    if line == "PART":
        return False
    if "http" in line:
        return False
    if re.match(r"^\d+\)", line):
        return False
    if re.match(r"^\d+\s+.+", line) and "?" in line:
        return True
    if "?" in line:
        question_position = line.find("?")
        after_question = line[
            question_position +1:
        ].strip()

        if len(after_question) <= 3:
            return True
    return False

# 자꾸 소제목에 숫자랑 ? 들어간 친구 나와서 추가한 코드

def merge_question_lines(lines):
    new_lines = []
    question_text = ""
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if question_text != "":
            question_text = question_text + " " + line
            if "?" in line:
                new_lines.append(question_text)
                question_text = ""
            continue
        if len(line) > 2:
            if line[0] in "123456789" and line[1] == " ":
                question_text = line
                if "?" in line:
                    new_lines.append(question_text)
                    question_text = ""
                continue
            new_lines.append(line)
    if question_text != "":
        new_lines.append(question_text)
    return new_lines

from langchain_core.documents import Document

section_docs = []

present_section = "개요"
present_text = []

present_part = None
present_chapter = None

start_page = None
end_page = None

for doc in docs:
    page = doc.metadata["page"] +1

    part = doc.metadata["part"]
    chapter = doc.metadata["chapter"]
    lines = doc.page_content.split("\n")
    lines = merge_question_lines(lines)

    if present_part is not None and part != present_part:
        if len(present_text) >0:
            section_docs.append(
                Document(
                    page_content="\n".join(present_text),

                    metadata={
                        "part": present_part,
                        "chapter": present_chapter,
                        "keyword": present_chapter,
                        "section": present_section,
                        "page_start": start_page,
                        "page_end": end_page,
                        "content_type": "text",
                    }
                )
            )
        present_text = []
        present_section = "개요"
        start_page = page

    present_part = part
    present_chapter = chapter

    if start_page == None:
        start_page = page

# 소제목 기준으로 자르기

    for line in lines :
        line = line.strip()

        if not line:
            continue

        if is_heading(line, chapter):
            if len(present_text) >0:
                section_docs.append(
                    Document(
                        page_content="\n".join(present_text),
                        metadata={
                        "part": present_part,
                        "chapter": present_chapter,
                        "keyword": present_chapter,
                        "section": present_section,
                        "page_start": start_page,
                        "page_end": end_page,
                        "content_type": "text",
                        }
                    )
                )

            present_text = []
            present_section = line
            start_page = page

        else:
            present_text.append(line)

            end_page = page

if len(present_text) >0:
    section_docs.append(
        Document(
            page_content="\n".join(present_text),

            metadata={
                "part": present_part,
                "chapter": present_chapter,
                "keyword": present_chapter,
                "section": present_section,
                "page_start": start_page,
                "page_end": end_page,
                "content_type": "text",
            }
        )
    )

In [38]:
# 표 구분

table_docs = []

for doc in docs:
    page = doc.metadata["page"] +1
    if page == 12 or page == 22:
        table_doc = Document(
            page_content=doc.page_content,

            metadata ={
                "part": doc.metadata["part"],
                "chapter": doc.metadata["chapter"],
                "keyword": doc.metadata["keyword"],
                "section": "표",
                "page_start": page,
                "page_end": page,
                "content_type": "table"
            }
        )

        table_docs.append(table_doc)


# 문서 + 표 합치기

rag_docs = section_docs + table_docs

In [39]:
# Chunking (큰 Document만 자르기)

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1200,
    chunk_overlap = 120
)

chunks = []

for doc in rag_docs:
    if doc.metadata["content_type"] == "table":
        chunks.append(doc)

    elif len(doc.page_content) <= 1200:
        chunks.append(doc)
    else:
        split_docs = text_splitter.split_documents(
            [doc]
        )

        for split_doc in split_docs:
            chunks.append(split_doc)

# 짧은 chunk 제거

final_chunks = []

for chunk in chunks:
    if(
        chunk.metadata["section"] == "개요"
        and len(chunk.page_content) < 100
    ) :
        continue
    final_chunks.append(chunk)

In [40]:
# 임베딩 만들기

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# metadata 붙이기

from langchain_core.documents import Document

embedding_docs = []

for doc in final_chunks:
    content = f"""
PART : {doc.metadata["part"]}
Chapter : {doc.metadata["chapter"]}
Section: {doc.metadata["section"]}

{doc.page_content}
"""
    new_doc = Document(
        page_content=content,
        metadata=doc.metadata
    )
    embedding_docs.append(new_doc)

In [ ]:
# VectorStore 만들기

from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(
    embedding=embeddings
)

vector_store.add_documents(
    documents=embedding_docs
)


# Retriever 만들기


retriever = vector_store.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs={
        "k": 5,
        "score_threshold": 0.4
    }
)


NameError: name 'embeddings' is not defined

In [ ]:
# llm 연결하기

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1
)

# 프롬프트

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    자, 테스트의 답변이다. 아래 형식으로 질문에 답해라.
    그러나, 만에 하나 문서에서 답을 도저히 못 찾겠으면 "미안합니다." 라고 답해라.
    그리고, 훌륭한 대학원생 답게 당연하게도 답변에 참고한 PART, Chapter, Section, Page는 주석으로 달도록.

    [문서]
    {context}


    [질문]
    {question}


    [답변]
"""
)

# document 모으기

def format_docs(docs):

    context = ""

    for doc in docs:

        context += f"""
PART: {doc.metadata["part"]}
Chapter: {doc.metadata["chapter"]}
Section: {doc.metadata["section"]}
Page: {doc.metadata["page_start"]} ~ {doc.metadata["page_end"]}

{doc.page_content}

"""

    return context

# Parser 연결

from langchain_core.output_parsers import StrOutputParser

chain = (
    prompt
    | llm
    | StrOutputParser()
)

In [48]:
question = "생성형 AI가 만든 결과물에 대해 인간이 저작권을 가질 수 있는 경우는 어떤 상황인지 사례를 들어 설명하세요."

# 관련 문서 검색
retrieved_docs = retrieve_docs(question)


# 검색된 문서를 하나의 문자열로 변환
context = format_docs(retrieved_docs)


# LLM에게 질문
response = chain.invoke(
    {
        "context": context,
        "question": question
    }
)


print(response)


생성형 AI가 만든 결과물에 대해 인간이 저작권을 가질 수 있는 경우는, 인간이 AI가 생성한 결과물에 창작적 표현을 추가했을 때입니다. 예를 들어, 사용자가 생성형 AI에게 특정한 주제나 스타일에 대한 구체적인 지시를 내리고, 그 결과물에 자신의 독창적인 아이디어나 감정을 표현하여 수정하거나 보완한 경우입니다. 이 경우, 사용자는 AI가 생성한 결과물에 대해 자신의 창작적 기여 부분에 대해 저작권을 인정받을 수 있습니다.

예를 들어, 사용자가 생성형 AI에게 "여름 바다의 풍경을 그려줘"라고 요청한 후, AI가 생성한 그림을 바탕으로 사용자가 추가적으로 색상이나 구도를 수정하여 자신만의 스타일로 완성한 경우, 이 수정된 결과물에 대해서는 사용자가 저작권을 가질 수 있습니다. 

이와 관련된 법적 해석에 따르면, 저작권법 제2조에 명시된 바와 같이 '저작물'은 인간의 사상 또는 감정을 표현한 창작물이어야 하며, 따라서 인간의 창작적 기여가 인정되는 경우에만 저작권이 부여될 수 있습니다(참고: PART 02, Chapter: 저작권, Section: 1, Page: 6 ~ 7).
